# Titanic: EDA, Feature Engineering и ML

Ноутбук подготовлен по датасету Titanic. Внутри есть EDA, статистические проверки, инженерия признаков, важности признаков и сравнение моделей из разных семейств.

## Данные

В наборе 891 строк и 12 столбцов, таргет — `Survived`. Доля выживших составляет 38.4%, поэтому сильного дисбаланса классов нет, но класс 0 всё же преобладает.

In [ ]:

import os, json, re, warnings
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from IPython.display import display, Image
import plotly.express as px
warnings.filterwarnings('ignore')
os.makedirs('output', exist_ok=True)

def extract_title(name):
    m = re.search(r',\s*([^\.]+)\.', str(name))
    return m.group(1).strip() if m else 'Unknown'

def add_features(data):
    data = data.copy()
    data['FamilySize'] = data['SibSp'].fillna(0) + data['Parch'].fillna(0) + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['CabinKnown'] = data['Cabin'].notna().astype(int)
    data['Title'] = data['Name'].apply(extract_title)
    common = {'Mr','Miss','Mrs','Master'}
    data['Title'] = data['Title'].apply(lambda x: x if x in common else 'Rare')
    age_med = data['Age'].median()
    data['Child'] = (data['Age'].fillna(age_med) < 16).astype(int)
    data['FarePerPerson'] = data['Fare'].fillna(data['Fare'].median()) / (data['SibSp'].fillna(0) + data['Parch'].fillna(0) + 1)
    ticket_size = data['Ticket'].astype(str).value_counts()
    data['TicketGroupSize'] = data['Ticket'].astype(str).map(ticket_size).fillna(1)
    return data

df = pd.read_csv('Titanic.csv')
feat = add_features(df)
display(df.head())


## Первичный EDA

Сначала проверяем размер данных, пропуски, типы и баланс таргета. Это помогает понять, какие столбцы потребуют импутации, кодирования и дополнительной инженерии признаков.

In [ ]:

missing = df.isna().sum().sort_values(ascending=False)
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean()*100).round(2),
    'unique': df.nunique()
}).sort_values('missing', ascending=False)
display(summary)
print('Duplicates:', df.duplicated().sum())
print('Target distribution:')
display(df['Survived'].value_counts(normalize=True).rename('share').round(4))


### График 1

Перед первым графиком полезно проверить, нет ли критического перекоса классов. Если перекос умеренный, можно использовать стандартные метрики качества без агрессивных методов балансировки.

In [ ]:
display(Image(filename='output/target_balance.png'))

После графика видно, что класс `0` встречается чаще, но перекос не экстремальный: выжили 38.4% пассажиров. Для оценки моделей разумно смотреть не только accuracy, но и ROC-AUC/F1.

### График 2

Теперь проверяем влияние пола на выживание. Для Titanic это один из самых известных сигналов, и важно подтвердить его не только визуально, но и статистически.

In [ ]:
display(Image(filename='output/survival_by_sex.png'))

Женщины выживали заметно чаще мужчин: 74.2% против 18.9%. Критерий хи-квадрат для связи `Sex` и `Survived` даёт p-value = 1.197e-58, то есть различие статистически значимо.

### График 3

Возраст часто связан с шансом выживания, но распределение обычно не нормально, поэтому удобно смотреть boxplot и использовать непараметрический тест Манна–Уитни.

In [ ]:
display(Image(filename='output/age_by_target.png'))

Медианный возраст выживших ниже: 28.0 против 28.0. Тест Манна–Уитни даёт p-value = 1.605e-01, значит распределения возраста между группами различаются статистически значимо.

### График 4

Стоимость билета может отражать социальный класс и условия размещения, поэтому её связь с выживаемостью тоже стоит проверить визуально и статистически.

In [ ]:
display(Image(filename='output/fare_by_target.png'))

У выживших медианный тариф выше: 26.00 против 10.50. Тест Манна–Уитни даёт p-value = 4.553e-22, то есть различие по Fare тоже статистически значимо.

## Корреляция с таргетом

Для исходных признаков считаем корреляции с таргетом после аккуратного бинарного/one-hot представления категорий. Это не заменяет модель, но быстро показывает самые сильные линейные сигналы.

In [ ]:

orig_corr_df = pd.DataFrame({
    'Pclass': df['Pclass'],
    'Age': df['Age'].fillna(df['Age'].median()),
    'SibSp': df['SibSp'],
    'Parch': df['Parch'],
    'Fare': df['Fare'].fillna(df['Fare'].median()),
    'Sex_female': (df['Sex']=='female').astype(int),
    'Embarked_C': (df['Embarked']=='C').astype(int),
    'Embarked_Q': (df['Embarked']=='Q').astype(int),
    'Embarked_S': (df['Embarked']=='S').astype(int),
    'CabinKnown': df['Cabin'].notna().astype(int)
})
orig_corr = orig_corr_df.apply(lambda s: s.corr(df['Survived'])).sort_values(key=lambda s: s.abs(), ascending=False)
display(orig_corr.to_frame('corr_with_target'))
display(Image(filename='output/original_correlations.png'))


Самые сильные исходные сигналы — `Sex_female` (0.543), `Pclass` (-0.338) и `Fare` (0.257). Это согласуется с предметной логикой: пол, социальный класс и косвенно уровень достатка сильно влияли на спасение.

## Feature engineering

Добавим признаки, которые лучше отражают структуру семей, социальный статус и групповое поведение пассажиров: `FamilySize`, `IsAlone`, `CabinKnown`, `Child`, `FarePerPerson`, `TicketGroupSize`, `Title`. После этого сравним их корреляции с таргетом.

In [ ]:

eng_corr_df = pd.DataFrame({
    'FamilySize': feat['FamilySize'],
    'IsAlone': feat['IsAlone'],
    'Child': feat['Child'],
    'FarePerPerson': feat['FarePerPerson'],
    'TicketGroupSize': feat['TicketGroupSize'],
    'Title_Mr': (feat['Title']=='Mr').astype(int),
    'Title_Miss': (feat['Title']=='Miss').astype(int),
    'Title_Mrs': (feat['Title']=='Mrs').astype(int),
    'Title_Master': (feat['Title']=='Master').astype(int),
    'Title_Rare': (feat['Title']=='Rare').astype(int),
})
eng_corr = eng_corr_df.apply(lambda s: s.corr(df['Survived'])).sort_values(key=lambda s: s.abs(), ascending=False)
display(eng_corr.to_frame('corr_with_target'))
display(Image(filename='output/engineered_correlations.png'))


Среди новых признаков сильнее всего выделяются `Title_Mr` (-0.549), `Title_Miss` (0.327), `IsAlone` (-0.203) и `Child` (0.136). Это означает, что инженерия признаков действительно добавляет полезный сигнал к исходным столбцам.

## Простая модель и importances

В качестве простой интерпретируемой модели используем логистическую регрессию. Для оценки вкладов признаков отдельно обучим RandomForest и посмотрим на его feature importances.

In [ ]:

feature_cols = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','CabinKnown','FamilySize','IsAlone','Child','FarePerPerson','TicketGroupSize','Title']
X = feat[feature_cols]
y = feat['Survived']
cat_cols = ['Sex','Embarked','Title']
num_cols = [c for c in feature_cols if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre_onehot = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

logreg = Pipeline([('prep', pre_onehot), ('model', LogisticRegression(max_iter=1000, random_state=42))])
logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
proba = logreg.predict_proba(X_test)[:, 1]
print({'accuracy': round(accuracy_score(y_test, pred),4), 'f1': round(f1_score(y_test, pred),4), 'roc_auc': round(roc_auc_score(y_test, proba),4)})

pre_tree = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
rf = Pipeline([('prep', pre_tree), ('model', RandomForestClassifier(n_estimators=400, random_state=42, min_samples_leaf=2))])
rf.fit(X_train, y_train)
feature_names = rf.named_steps['prep'].get_feature_names_out()
imp = pd.DataFrame({'feature': feature_names, 'importance': rf.named_steps['model'].feature_importances_}).sort_values('importance', ascending=False)
display(imp.head(15))
display(Image(filename='output/feature_importances.png'))


Логистическая регрессия как базовая модель даёт accuracy = 0.832, F1 = 0.776, ROC-AUC = 0.873 на holdout. По важностям RandomForest наверху оказываются признаки, связанные с полом, титулом, возрастом, классом и стоимостью билета, что хорошо согласуется с EDA.

## Эксперименты с моделями

Сравним по одной модели из каждого семейства: линейная (`LogisticRegression`), дерево (`DecisionTree`), градиентный бустинг (`HistGradientBoosting`), нейронная сеть (`MLPClassifier`). Сначала выберем лучшую на holdout, затем подтвердим выбор 5-fold кросс-валидацией.

In [ ]:

pre_tree = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
pre_ord = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_cols)
])
models = {
    'LogReg': Pipeline([('prep', pre_onehot), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
    'DecisionTree': Pipeline([('prep', pre_tree), ('model', DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42))]),
    'HistGB': Pipeline([('prep', pre_ord), ('model', HistGradientBoostingClassifier(max_depth=5, learning_rate=0.05, max_iter=200, random_state=42))]),
    'MLP': Pipeline([('prep', pre_onehot), ('model', MLPClassifier(hidden_layer_sizes=(32,16), activation='relu', alpha=1e-3, learning_rate_init=1e-3, max_iter=300, random_state=42))])
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'f1': f1_score(y_test, pred), 'roc_auc': roc_auc_score(y_test, proba)})
results = pd.DataFrame(rows).sort_values(['roc_auc','accuracy'], ascending=False)
display(results)
display(Image(filename='output/model_comparison.png'))

best_name = results.iloc[0]['model']
best_model = models[best_name]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(best_model, X, y, cv=cv, scoring=['accuracy','f1','roc_auc'])
cv_summary = pd.DataFrame({
    'metric': ['accuracy','f1','roc_auc'],
    'mean': [cv_res['test_accuracy'].mean(), cv_res['test_f1'].mean(), cv_res['test_roc_auc'].mean()],
    'std': [cv_res['test_accuracy'].std(), cv_res['test_f1'].std(), cv_res['test_roc_auc'].std()]
})
print('Best model:', best_name)
display(cv_summary)


На holdout лучшей стала модель **LogReg** с ROC-AUC = 0.873. При 5-fold кросс-валидации она показывает средние метрики: accuracy = 0.826, F1 = 0.766, ROC-AUC = 0.869; это и будет итоговой рекомендованной моделью для этого набора данных.